In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt
from datetime import timedelta
import os

In [43]:
customer_files = pd.ExcelFile(os.path.join('data', 'loyalty_data.xlsx')).sheet_names
customer_files = customer_files[1:4]+customer_files[-1:]

def read_sheet(sheet_name):
    return pd.read_excel(os.path.join('data', 'loyalty_data.xlsx'), sheet_name=sheet_name)
loyalty_points,loyalty_benefits, transactions, merchants = [read_sheet(files) for files in customer_files]

In [4]:
merchants['Merchant'].nunique()

45

In [5]:
merchants['MerchantID'].nunique()

44

In [6]:
merchants['Merchant'].unique()

<StringArray>
[                   '7eleven',                  'Aeon Mall',
                      'Big C',                    'Bobapop',
                     'BSMART',             'Cà Phê Ông Bầu',
              'CHEESE COFFEE',                   'Circle K',
                  'Co.opmart',               'Coffee House',
                'Cộng Cà phê', 'Công ty TNHH Cà Phê Gemini',
               'Đá Đen Coffe',                   'E coffee',
                 'FAMILYMART',                   'FINELIFE',
                    'GongCha',                       'GS25',
                  'GUTA CAFE',                  'Highlands',
                    'Katinat',                    'Koi The',
                    'Koi Thé',                     'LAZADA',
                 'Lotte Mart',         'MEGA-PHINDELI CAFE',
              'Milano Coffee',                   'Ministop',
             'MM Mega Market',                   'Mobifone',
                     'Passio',                  'PHUC LONG',
          

Since the number of unique Merchant should be equal to the number of unique Merchant ID. I investigated the unique values of Merchant and noticed that VIETTEL and Viettel are the same company. So, I convert VIETTEL 
into Viettel.

In [7]:
merchants['Merchant'] = merchants['Merchant'].replace(to_replace='VIETTEL',value='Viettel')

In [8]:
transactions.dtypes

DATE                    datetime64[us]
Order_id                         int64
NEWVERTICAL_Merchant               str
MerchantID                       int64
User_id                          int64
GMV                              int64
Service Group                      str
Loyalty points                 float64
Rank                           float64
%cash back                     float64
dtype: object

# **PART 1. Data processing**

**Calculate daily loyalty points**

Combined with the 'Loyalty Points' table, add a column 'Loyalty Points' in 'Transactions' table with given rules. Then create another table named 'Loyalty Ranking' which must includes columns named Rank_name and Calculated_points to calculate the Rank of each user on daily basic. At the end of Mar 2022, how many user achived rank Gold?

In [44]:
transactions = transactions.reset_index()

In [45]:
loyalty_points = []
for index, row in transactions.iterrows():
    if row['Service Group'] in ['supermarket', 'marketplace', 'Coffee chains and Milk tea']:
        loyalty_points.append(min(row['GMV']/1000, 500))
    elif row['Service Group'] == 'data':
        loyalty_points.append(min(row['GMV']/100, 1000))
    elif row['Service Group'] in ['cvs','Offline Beverage']:
        loyalty_points.append(min(row['GMV']/1000,300))
transactions['Loyalty Points'] = loyalty_points

**How** **many user archived rank Gold at the end on March?**

Because after 30 days, loyalty points from transaction will be reset. So, the accumulated loyalty points at the end of March will start from March 2nd.

In [17]:
end_of_March_ranking = transactions[(transactions['DATE']>= '2022-03-02') & (transactions['DATE']< '2022-04-01')]
end_of_March_total_point = end_of_March_ranking.groupby('User_id')['Loyalty Points'].sum().reset_index()
end_of_March_total_point['Ranking'] = end_of_March_total_point.apply(lambda row: 'STANDARD' if row['Loyalty Points'] < 1000 else ('SILVER' if row['Loyalty Points'] < 2000 else 'GOLD' if row['Loyalty Points'] < 5000 else 'DIAMOND'), axis=1)
end_of_March_total_point['Ranking'].value_counts()

Ranking
STANDARD    1490
SILVER       294
GOLD         123
DIAMOND       14
Name: count, dtype: int64

**Calculate total cashback for February**

Combined with the 'Loyalty benefits' table and 'Loyalty Ranking' table, add columns '%cashback'  in 'Transactions' table and calculate the total cashback cost in February 2022.

In [34]:
transactions.head(5)

,index,DATE,Order_id,NEWVERTICAL_Merchant,MerchantID,User_id,GMV,Service Group,Loyalty points,Rank,%cash back,Loyalty Points
0,0,2021-01-01,8733622706,Marketplace,37,61386143,100000,marketplace,NaN,NaN,NaN,100.0
1,1,2021-01-01,8726857991,Supermarket,9,48453125,5000,supermarket,NaN,NaN,NaN,5.0
2,2,2021-01-01,8737326894,Supermarket,9,49921027,106600,supermarket,NaN,NaN,NaN,106.6
3,3,2021-01-01,8732579078,supermarket,9,46022523,270000,supermarket,NaN,NaN,NaN,270.0
4,4,2021-01-01,8725567343,CVS,8,44014594,68000,cvs,NaN,NaN,NaN,68.0


In [59]:
# Cach 1
def calculate_loyalty_point(user, date):
  user_transactions = transactions[transactions['User_id'] == user]
  upper_filter = user_transactions['DATE'] <= date
  lower_filter = user_transactions['DATE'] >= (date - timedelta(days=30))
  within_30d_transactions = user_transactions[upper_filter & lower_filter]
  accumulated_loyalty_points = within_30d_transactions['Loyalty Points'].sum() 
  return accumulated_loyalty_points
transactions_test0 = transactions.sort_values(['User_id', 'DATE'])
transactions_test0['30 days accumulated loyalty points'] = transactions_test0.apply(lambda row: calculate_loyalty_point(row['User_id'], row['DATE']), axis=1)

In [ ]:
# Cach 2
transactions_test = transactions.sort_values(['User_id', 'DATE'])
transactions_test = transactions_test.set_index('DATE')
transactions_test['30 days accumulated loyalty points'] = transactions_test.groupby('User_id')['Loyalty Points'].rolling('30D', closed='both').sum().reset_index(0,drop=True)
transactions_test = transactions_test.reset_index()

In [ ]:
# Cach 3 -- check lai
transactions_test1 = transactions.sort_values('DATE').reset_index(drop=True)
merged = pd.merge_asof(transactions_test1, transactions_test1, left_on='DATE', right_on='DATE', by='User_id', direction='backward', tolerance=pd.Timedelta('30D'))
transactions_test1['30 days accumulated loyalty points'] =  merged.groupby('index_x')['Loyalty Points_y'].transform('sum')
transactions_test1[transactions_test1['User_id'] == 61386143]